In [1]:
!pip install pandas==2.1.4 -qq
!pip install google-cloud-bigquery==3.40.0 -qq
!pip install openpyxl==3.1.2 -qq
!pip install db-dtypes==1.5.0 -qq
!pip install catboost -qq

In [2]:
import calendar
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from datetime import datetime
from google.cloud import bigquery
from zoneinfo import ZoneInfo
from matplotlib import pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [3]:
def generate_month_query(table_full_name: str, tz_name: str = "Asia/Ho_Chi_Minh", ref_date: datetime | None = None) -> str:
    """
    Trả về query cho "tháng hiện tại" theo timezone tz_name.
    table_full_name ví dụ: "real-estate-project-445516.real_estate_data.ads_data"
    ref_date dùng để test (nếu None thì lấy thời điểm hiện tại).
    """
    tz = ZoneInfo(tz_name)
    now = ref_date.astimezone(tz) if ref_date else datetime.now(tz)
    
    # year = now.year
    # month = now.month
    year = 2026
    month = 1

    # ngày bắt đầu tháng (00:00:00) và ngày cuối tháng (00:00:00)
    first_day_str = f"{year:04d}-{month:02d}-01T00:00:00"
    last_day_num = calendar.monthrange(year, month)[1]
    last_day_str = f"{year:04d}-{month:02d}-{last_day_num:02d}T00:00:00"

    query = f"""select *
                from `{table_full_name}`
                where (`Ngày đăng` <= '{first_day_str}' and `Ngày hết hạn` >= '{first_day_str}') or
                      (`Ngày đăng` >= '{first_day_str}' and `Ngày đăng` <= '{last_day_str}');"""
    return query


In [4]:
client = bigquery.Client.from_service_account_json("/lakehouse/default/Files/real-estate-project-445516-83dc50b692bc.json")

In [5]:
table = "real-estate-project-445516.real_estate_data.ads_data"
query = generate_month_query(table)

In [6]:
# df = client.query(query).to_dataframe()

In [7]:
df = pd.read_csv("/lakehouse/default/Files/ads_data_2026_01.csv")

In [8]:
df.columns

Index(['Mã tin', 'Ngày đăng', 'Ngày hết hạn', 'Loại tin', 'Loại quảng cáo',
       'Loại BĐS', 'Tỉnh thành phố', 'Quận', 'Địa chỉ 1', 'Địa chỉ 2',
       'Khoảng giá', 'Diện tích', 'Pháp lý', 'Link dự án', 'Tên dự án',
       'Số tầng lookup', 'Ngày gia hạn', 'Số phòng ngủ',
       'Số phòng tắm - vệ sinh', 'Nội thất', 'Số tầng', 'Mặt tiền',
       'Hướng nhà', 'Hướng ban công', 'Đường vào', 'Thời gian dự kiến vào ở',
       'Mức giá điện', 'Mức giá nước', 'Mức giá internet', 'Tiện ích',
       'Phường Xã Thị trấn', 'Tọa độ x', 'Tọa độ y'],
      dtype='object')

## Preprocessing

In [9]:
def safe_mean_round(s):
    values = [x for x in s if pd.notna(x)]
    return round(float(np.nanmean(values)), 2) if values else None

In [10]:
# bỏ những dòng không có giá
df1 = df.dropna(subset=['Khoảng giá']).copy()
df1 = df1[(df1["Khoảng giá"] >= 100000000) | (df1["Loại quảng cáo"] != "Bán")]

In [18]:
group_cols = [ 
    'Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận',  'Địa chỉ 1',
    'Diện tích', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Tên dự án', 
    'Hướng nhà', 'Hướng ban công', 'Số tầng', 'Mặt tiền', 'Đường vào',
]

agg_df = (
    df1
    .groupby(group_cols, dropna=False)
    .agg(
        mean_unique_khoang_gia = ('Khoảng giá', lambda s: np.mean(np.unique(s)) if len(np.unique(s))>0 else np.nan),
        phap_ly=("Pháp lý", "first"),
        noi_that=("Nội thất", "first")
    )
    .reset_index()
)

# nếu muốn chuyển mean sang triệu hoặc tỷ (ví dụ sang triệu VND):
agg_df['mean_unique_khoang_gia_million'] = agg_df['mean_unique_khoang_gia'] / 1e6

In [21]:
agg_df.shape

(385235, 18)

In [15]:
# làm tròn chữ cái đầu tiên của tất cả các giá trị trong cột bds_type
agg_df['Loại BĐS'] = agg_df['Loại BĐS'].str.strip().str.capitalize()

# rename loại bds
mapping = {
    "Nhà biệt thự, liền kề": "Nhà biệt thự / Nhà liền kề",
    "Nhà mặt phố": "Nhà phố",
    "Nhà riêng": "Nhà ở",
    "Nhà trọ, phòng trọ": "Nhà trọ"
}
agg_df["Loại BĐS"] = agg_df["Loại BĐS"].replace(mapping)

# rename pháp lý
mask = agg_df["phap_ly"].notna() & (agg_df["phap_ly"] != "Hợp đồng mua bán")
agg_df.loc[mask, "phap_ly"] = "Sổ đỏ/ Sổ hồng"

# rename nội thất
col_lower = agg_df["noi_that"].str.lower()
agg_df["noi_that"] = np.select(
    [
        col_lower.str.contains("cơ bản", na=False),
        col_lower.str.contains("không nội thất", na=False),
        col_lower.str.contains("đầy đủ", na=False),
    ],
    [
        "cơ bản",
        "không nội thất",
        "đầy đủ",
    ],
    default=np.where(agg_df["noi_that"].notna(), "đầy đủ", "NaN")
)

agg_df = agg_df[agg_df["Loại BĐS"].isin(["Nhà phố", "Nhà ở", "Căn hộ chung cư", "Đất"])]


In [16]:
agg_df["so_tang"] = agg_df["Số tầng"].str.extract(r'(\d+)')
agg_df["so_tang"] = pd.to_numeric(agg_df["so_tang"], errors='coerce')

agg_df["mat_tien"] = agg_df["Mặt tiền"].str.extract(r'(\d+)')
agg_df["mat_tien"] = pd.to_numeric(agg_df["mat_tien"], errors='coerce')

agg_df["duong_vao"] = agg_df["Đường vào"].str.extract(r'(\d+)')
agg_df["duong_vao"] = pd.to_numeric(agg_df["duong_vao"], errors='coerce')

In [17]:
agg_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 307420 entries, 1929 to 370322
Data columns (total 21 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Loại quảng cáo                  307420 non-null  object 
 1   Loại BĐS                        307420 non-null  object 
 2   Tỉnh thành phố                  307420 non-null  object 
 3   Quận                            307420 non-null  object 
 4   Địa chỉ 1                       307420 non-null  object 
 5   Diện tích                       307420 non-null  float64
 6   Số phòng ngủ                    182705 non-null  float64
 7   Số phòng tắm - vệ sinh          170328 non-null  float64
 8   Tên dự án                       65128 non-null   object 
 9   Hướng nhà                       86011 non-null   object 
 10  Hướng ban công                  49563 non-null   object 
 11  Số tầng                         162970 non-null  object 
 12  Mặt tiền          

## Loại bỏ nhiễu

In [17]:
ban_df = agg_df[agg_df["Loại quảng cáo"] == "Bán"]
thue_df = agg_df[agg_df["Loại quảng cáo"] == "Cho thuê"]

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 19, Finished, Available, Finished)

In [19]:
price_col = 'mean_unique_khoang_gia_million'
group_cols = ['Loại BĐS', 'Tỉnh thành phố', 'Quận']

# thu thập index các dòng cần drop
idx_to_drop = []
for _, g in ban_df.groupby(group_cols):
    n = len(g)
    # luôn bỏ ít nhất 1 dòng
    k = max(1, int(np.ceil(n*0.01)))
    if k > 0:
        # chọn k dòng có giá cao nhất trong nhóm
        idx_to_drop.extend(g.nlargest(k, price_col).index.tolist())

# dataframe sau khi đã loại bỏ 1% hàng top theo mỗi nhóm
ban_df = ban_df.drop(index=idx_to_drop).reset_index(drop=True)

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 21, Finished, Available, Finished)

In [20]:
# df2 = ban_df[(ban_df["Tỉnh thành phố"] == "Hà Nội") & (ban_df["Loại BĐS"] == "Nhà ở")]
df2 = ban_df[(ban_df["Tỉnh thành phố"] == "Hà Nội")]
# df2 = ban_df

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 22, Finished, Available, Finished)

In [21]:
ban_df.columns

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 23, Finished, Available, Finished)

Index(['Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận',
       'Phường Xã Thị trấn', 'Địa chỉ 1', 'Diện tích', 'Số phòng ngủ',
       'Số phòng tắm - vệ sinh', 'Tên dự án', 'Hướng nhà', 'Hướng ban công',
       'Số tầng', 'Mặt tiền', 'Đường vào', 'mean_unique_khoang_gia', 'phap_ly',
       'noi_that', 'mean_unique_khoang_gia_million', 'so_tang', 'mat_tien',
       'duong_vao'],
      dtype='object')

In [22]:
# Separate the features (X) and the target (y)
X = df2.drop(columns=["mean_unique_khoang_gia", 'mean_unique_khoang_gia_million', "Số tầng", "Mặt tiền", "Đường vào"])
y = df2['mean_unique_khoang_gia_million']

# Split the data into a test subset (smaller portion) and a training subset (larger portion)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 24, Finished, Available, Finished)

In [23]:
X.columns

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 25, Finished, Available, Finished)

Index(['Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận',
       'Phường Xã Thị trấn', 'Địa chỉ 1', 'Diện tích', 'Số phòng ngủ',
       'Số phòng tắm - vệ sinh', 'Tên dự án', 'Hướng nhà', 'Hướng ban công',
       'phap_ly', 'noi_that', 'so_tang', 'mat_tien', 'duong_vao'],
      dtype='object')

## Tree-based

In [25]:
numerical_columns = X.select_dtypes(exclude = 'object').columns
categorical_columns = X_train.select_dtypes(include = 'object').columns
categorical_column_indices = [X_train.columns.get_loc(col) for col in categorical_columns]

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 27, Finished, Available, Finished)

In [26]:
# Convert NaN values to strings in the specified categorical columns
for col_index in categorical_column_indices:
    X_train.iloc[:, col_index] = X_train.iloc[:, col_index].astype(str).fillna('NaN')
    X_test.iloc[:, col_index] = X_test.iloc[:, col_index].astype(str).fillna('NaN')

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 28, Finished, Available, Finished)

In [27]:
# Additional regressors for models_with_categorical
models_with_categorical = [
    # DecisionTreeRegressor(),
    # RandomForestRegressor(max_depth=8, random_state=42),
    # RandomForestRegressor(),
    # GradientBoostingRegressor(learning_rate=0.03, max_depth=10, random_state=42),
    # XGBRegressor(learning_rate=0.03, max_depth=10, random_state=42),
    # XGBRegressor(),
    # SVR(),
    CatBoostRegressor(iterations=2995,  # Specify the number of boosting iterations
                      learning_rate=0.08222472905612799,  # Learning rate
                      depth = 11,
                      verbose=100,  # Print progress every 100 iterations
                      cat_features=categorical_column_indices,
                      l2_leaf_reg=4.36933961863642,
                      random_strength=3.2510914157888672e-09,
                      bagging_temperature=3.221967027448404
    )
    # CatBoostRegressor()  
]

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 29, Finished, Available, Finished)

In [28]:
# # # Encode input
# # X_encoded = X.copy()
# # for feature in categorical_columns:
# #   label_encoder = LabelEncoder()
# #   X_encoded[feature] = label_encoder.fit_transform(X_encoded[feature])

# # X_train_encoded, X_test_encoded, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)
# results_df = pd.DataFrame()

# for model in models_with_categorical:
#     # Fit the model
#     model.fit(X_train, y_train)

#     # Predict on the test set
#     y_pred = model.predict(X_test)

#     # Calculate metrics
#     mse = mean_squared_error(y_test, y_pred)
#     r2 = r2_score(y_test, y_pred)

#     # Append results to the dataframe
#     results_df = pd.concat([
#         results_df,
#         pd.DataFrame([{
#             'Model': type(model).__name__,
#             'MSE': mse,
#             'R2_Score': r2
#         }])
#     ], ignore_index=True)

    
#     df_eval = X_test[["Loại BĐS"]].copy()
#     df_eval["y_true"] = y_test.values
#     df_eval["y_pred"] = y_pred

#     r2_by_bds_type = (
#         df_eval
#         .groupby("Loại BĐS")
#         .apply(lambda x: r2_score(x["y_true"], x["y_pred"]))
#     )

#     print(r2_by_bds_type)

# # # Display the results dataframe
# # results_df

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 30, Finished, Available, Finished)

In [29]:
# df_eval = X_test[["Loại BĐS"]].copy()
# df_eval["y_true"] = y_test.values
# df_eval["y_pred"] = y_pred

# r2_by_bds_type = (
#     df_eval
#     .groupby("Loại BĐS")
#     .apply(lambda x: r2_score(x["y_true"], x["y_pred"]))
# )

# print(r2_by_bds_type)

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 31, Finished, Available, Finished)

In [30]:
# # Tránh chia cho 0
# mask = y_test != 0
# mape = (
#     np.mean(
#         np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])
#     ) * 100
# )

# results_df = pd.concat([
#     results_df,
#     pd.DataFrame([{
#         'Model': type(model).__name__,
#         'MSE': mse,
#         'R2_Score': r2,
#         'MAPE_%': mape
#     }])
# ], ignore_index=True)
# print(results_df)

# mape_by_bds_type = (
#     df_eval[df_eval["y_true"] != 0]
#     .groupby("Loại BĐS")
#     .apply(lambda x: np.mean(
#         np.abs((x["y_true"] - x["y_pred"]) / x["y_true"])
#     ) * 100)
# )
# print(mape_by_bds_type)

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 32, Finished, Available, Finished)

In [31]:
# test_cases = pd.DataFrame([{
#     'Loại quảng cáo': 'Bán',
#     'Loại BĐS': 'Căn hộ chung cư',
#     'Tỉnh thành phố': 'Hà Nội',
#     'Quận': 'Hoàn Kiếm',
#     'Địa chỉ 1': 'phố Hai Bà Trưng',
#     'Diện tích': 112,
#     'Số phòng ngủ': np.nan,
#     'Số phòng tắm - vệ sinh': np.nan,
#     'Tên dự án': 'NaN',
#     'Hướng nhà': 'Đông - Bắc',
#     'Hướng ban công': 'Đông - Bắc',
#     'phap_ly': 'Sổ đỏ/ Sổ hồng',
#     'noi_that': 'NaN',
#     'so_tang': 5,
#     'mat_tien': 4.3,
#     'duong_vao': 40
# }])

# model.predict(test_cases)

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 33, Finished, Available, Finished)

In [32]:
# r2_by_province

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 34, Finished, Available, Finished)

In [33]:
import optuna

def objective(trial):

    params = {
        "iterations": trial.suggest_int("iterations", 500, 3000),
        "depth": trial.suggest_int("depth", 8, 16),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.5, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-9, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 5),
        "loss_function": "RMSE",
        "verbose": 0,
    }

    model = CatBoostRegressor(**params)

    model.fit(
        X_train, y_train,
        eval_set=(X_test, y_test),
        cat_features=categorical_column_indices,
        early_stopping_rounds=100,
        verbose=False
    )

    preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    print(f"Trial {trial.number} | RMSE: {rmse:.4f} | R2: {r2:.4f}")

    return rmse


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 35, Finished, Available, Finished)

[I 2026-02-27 04:16:38,906] A new study created in memory with name: no-name-dfe0ec94-c4ce-4555-af35-3f8618859eba
[I 2026-02-27 05:39:54,986] Trial 18 finished with value: 12570.856854901785 and parameters: {'iterations': 2024, 'depth': 11, 'learning_rate': 0.015597189717460316, 'l2_leaf_reg': 0.02766461199698807, 'random_strength': 1.426979160777212e-07, 'bagging_temperature': 3.287364233303059}. Best is trial 17 with value: 12276.240182917989.
[I 2026-02-27 05:44:08,677] Trial 24 finished with value: 13510.287235481657 and parameters: {'iterations': 2328, 'depth': 12, 'learning_rate': 0.48929977808262926, 'l2_leaf_reg': 3.625356491131225, 'random_strength': 2.2766767591766962e-08, 'bagging_temperature': 3.0665201103702064}. Best is trial 19 with value: 12065.94176698329.


Trial 0 | RMSE: 17072.0821 | R2: 0.8065
Trial 1 | RMSE: 15809.2181 | R2: 0.8341
Trial 2 | RMSE: 18033.0127 | R2: 0.7841
Trial 3 | RMSE: 15697.6193 | R2: 0.8364
Trial 4 | RMSE: 12376.5767 | R2: 0.8983
Trial 5 | RMSE: 13529.7154 | R2: 0.8785
Trial 6 | RMSE: 18444.5634 | R2: 0.7741
Trial 7 | RMSE: 14658.1183 | R2: 0.8574
Trial 8 | RMSE: 18480.7886 | R2: 0.7733
Trial 9 | RMSE: 15437.8218 | R2: 0.8418
Trial 10 | RMSE: 12774.9355 | R2: 0.8917
Trial 11 | RMSE: 12442.6821 | R2: 0.8972
Trial 12 | RMSE: 12482.9203 | R2: 0.8965
Trial 13 | RMSE: 12692.2481 | R2: 0.8931
Trial 14 | RMSE: 12444.8606 | R2: 0.8972
Trial 15 | RMSE: 13041.1141 | R2: 0.8871
Trial 16 | RMSE: 12469.6282 | R2: 0.8968
Trial 17 | RMSE: 12276.2402 | R2: 0.8999
Trial 18 | RMSE: 12570.8569 | R2: 0.8951
Trial 19 | RMSE: 12065.9418 | R2: 0.9033
Trial 20 | RMSE: 13836.5620 | R2: 0.8729
Trial 21 | RMSE: 12601.4843 | R2: 0.8946
Trial 22 | RMSE: 12331.5968 | R2: 0.8990
Trial 23 | RMSE: 12311.5590 | R2: 0.8994
Trial 24 | RMSE: 13510.287

In [34]:
# model = CatBoostRegressor(iterations=1000,  # Specify the number of boosting iterations
#                           learning_rate=0.1,  # Learning rate
#                           depth = 16,
#                           verbose=100,  # Print progress every 100 iterations
#                           random_seed=42,
#                           cat_features=categorical_column_indices)

# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

# r2_score(y_test, y_pred)

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 36, Finished, Available, Finished)

In [35]:

# # Get feature importances
# feature_importance = model.get_feature_importance()

# # Get feature names
# feature_names = X_train.columns  # Replace with the actual column names of your features

# # Create a dictionary to store feature names and their corresponding importance scores
# feature_importance_dict = dict(zip(feature_names, feature_importance))

# # Sort feature importances and select the top 10
# top_features = sorted(feature_importance_dict.items(), key=lambda x: x[1], reverse=True)[:10]

# # Print the top 10 most important features
# print("Top 10 Most Important Features:")
# for feature, importance in top_features:
#     print(f"{feature}: {importance}")

# # Plot only the top 10 feature importances in descending order
# top_feature_names, top_feature_importance = zip(*top_features)

# plt.barh(top_feature_names[::-1], top_feature_importance[::-1])  # Reverse the order for descending plot
# plt.xlabel('Feature Importance Score')
# plt.ylabel('Features')
# plt.title('Top 10 Most Important Features')
# plt.show()
# ## Randomly select 4000, 5000, 6000 samples from the dataset
# ### 2000 samples
# def training(df):

#     # Separate the features (X) and the target (y)
#     X = df.drop(columns=['ad_price'])
#     y = df['ad_price']

#     # Split the data into a test subset (smaller portion) and a training subset (larger portion)
#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#     numerical_columns = X.select_dtypes(exclude = 'object').columns
#     categorical_columns = X_train.select_dtypes(include = 'object').columns
#     categorical_column_indices = [X_train.columns.get_loc(col) for col in categorical_columns]

#     # Convert NaN values to strings in the specified categorical columns
#     for col_index in categorical_column_indices:
#         X_train.iloc[:, col_index] = X_train.iloc[:, col_index].astype(str).fillna('NaN')
#         X_test.iloc[:, col_index] = X_test.iloc[:, col_index].astype(str).fillna('NaN')

#     # Initialize an empty dataframe to store results
#     results_df = pd.DataFrame(columns=['Model', 'MSE', 'R2_Score'])

#     # Additional regressors for models_with_categorical
#     models_with_categorical = [
#         DecisionTreeRegressor(),
#         RandomForestRegressor(max_depth=7, random_state=42),
#         GradientBoostingRegressor(),
#         XGBRegressor(learning_rate=0.2, max_depth=5, random_state=42),
#         SVR(),
#         CatBoostRegressor(iterations=1000,  # Specify the number of boosting iterations
#                           learning_rate=0.1,  # Learning rate
#                           depth = 7,
#                           verbose=100,  # Print progress every 100 iterations
#                           random_seed=42,
#                           cat_features=categorical_column_indices)
#     ]

#     # Additional regressors for models_without_categorical
#     models_without_categorical = [
#         LinearRegression(),
#         Ridge(),
#         Lasso(),
#         KNeighborsRegressor()
#     ]
#     for model in models_without_categorical:
#             # Fit the model
#             model.fit(X_train[numerical_columns].fillna(X[numerical_columns].mean()), y_train)

#             # Predict on the test set
#             y_pred = model.predict(X_test[numerical_columns].fillna(X[numerical_columns].mean()))

#             # Calculate metrics
#             mse = mean_squared_error(y_test, y_pred)
#             r2 = r2_score(y_test, y_pred)

#             # Append results to the dataframe
#             results_df = results_df.append({'Model': type(model).__name__,
#                                             'MSE': mse,
#                                             'R2_Score': r2},
#                                           ignore_index=True)
#     # Encode input
#     X_encoded = X.copy()
#     for feature in categorical_columns:
#       label_encoder = LabelEncoder()
#       X_encoded[feature] = label_encoder.fit_transform(X_encoded[feature])

#     X_train_encoded, X_test_encoded, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

#     for model in models_with_categorical:
#             # Fit the model
#             model.fit(X_train_encoded.fillna(X_encoded.mean()), y_train)

#             # Predict on the test set
#             y_pred = model.predict(X_test_encoded.fillna(X_encoded.mean()))

#             # Calculate metrics
#             mse = mean_squared_error(y_test, y_pred)
#             r2 = r2_score(y_test, y_pred)

#             # Append results to the dataframe
#             results_df = results_df.append({'Model': type(model).__name__,
#                                             'MSE': mse,
#                                             'R2_Score': r2},
#                                           ignore_index=True)
#     # Display the results dataframe
#     print(results_df)
# df_2000 = df2.sample(2000)
# training(df_2000)
# ### 5000 samples
# df_5000 = df2.sample(5000)
# training(df_5000)
# ### 8000 samples
# df_8000 = df2.sample(8000)
# training(df_8000)

StatementMeta(, f8653958-e446-431e-b369-8279a8e9abbf, 37, Finished, Available, Finished)